# MarketMosaic Week 1: Exploratory Data Analysis

This notebook is a reproducible, beginner-friendly walkthrough of the simulated Google Meridian marketing-mix panel. Run the cells from top to bottom. The printed tables are the numerical evidence used in `reports/data_audit.md`.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data/raw/geo_all_channels.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from eda import (load_and_prepare_data, data_quality_report,
                 descriptive_statistics, channel_summary,
                 geography_summary, temporal_summary,
                 correlation_matrix, plot_eda_figures,
                 SPEND_COLUMNS, IMPRESSION_COLUMNS)

DATA_PATH = PROJECT_ROOT / 'data/raw/geo_all_channels.csv'
FIGURE_PATH = PROJECT_ROOT / 'reports/figures'
data = load_and_prepare_data(DATA_PATH)
data.head()

## 1. Dataset overview

Each row is one geography-week observation. Channel columns describe paid media activity; controls describe external context; conversions are the outcome; and population provides market-size context.

In [ ]:
print('Rows and columns:', data.shape[0], data.shape[1] - 1, '(raw columns; time_dt is an analysis helper)')
print('Geographies:', data['geo'].nunique())
print('Time periods:', data['time_dt'].nunique())
print('Date range:', data['time_dt'].min().date(), 'to', data['time_dt'].max().date())
print('\nColumn types:\n', data.dtypes)

## 2. Data quality

The checks below explicitly test completeness, duplication, date parsing, panel uniqueness, and whether `Unnamed: 0` is an accidental exported row index.

In [ ]:
quality = data_quality_report(data)
for name, value in quality.items():
    if name not in {'rows_per_geo', 'periods_per_geo'}:
        print(f'{name}: {value}')
print('Rows per geography:', sorted(set(quality['rows_per_geo'].values())))
print('Periods per geography:', sorted(set(quality['periods_per_geo'].values())))

## 3. Statistical findings

`describe()` supplies count, mean, standard deviation, quartiles, minimum, and maximum. The median (50%) is often more representative than the mean when a variable is skewed.

In [ ]:
important_columns = SPEND_COLUMNS + ['conversions', 'revenue_per_conversion', 'Promo', 'competitor_sales_control', 'sentiment_score_control', 'population']
descriptive_statistics(data, important_columns).round(6)

## 4. Channel-level findings and correlations

These are Pearson correlations: they describe linear co-movement, not causal lift. Channel spend can be correlated with conversions because effective media, geography size, seasonality, and targeting may move together.

In [ ]:
channel_summary(data).round(6)

In [ ]:
correlation_matrix(data).round(4)

## 5. Geography and temporal findings

In [ ]:
geo = geography_summary(data)
print('Highest mean conversions')
display(geo.head(10).round(2))
print('Lowest mean conversions')
display(geo.tail(10).sort_values('conversions_mean').round(2))
weekly = temporal_summary(data)
print('Highest-conversion weeks')
display(weekly.nlargest(10, 'conversions_total').round(2))
print('Year-level means')
display(data.assign(year=data['time_dt'].dt.year).groupby('year')[['conversions'] + SPEND_COLUMNS].mean().round(2))

## 6. Visualisations

The helper creates and saves five named figures: distributions, channel-spend comparison, spend-versus-conversions scatterplots, geography conversions, and weekly conversions. Each chart is explained in the report.

In [ ]:
figures = plot_eda_figures(data, FIGURE_PATH)
print('Saved:', sorted(str(path.name) for path in FIGURE_PATH.glob('*.png')))
plt.show()